## Plan

#### 1. Preprocess and split

Use Lily’s code to:

- load breath-cycle data
- clean / encode / construct features
- split into train_data and test_data

#### 2. Add MFCC features

For each breath-cycle audio file:

- calculate MFCC features
- merge them onto train_data and test_data

Important: compute features separately on train and test files, don’t reshuffle after split

#### 3. Train + evaluate model

- Pass train_data and test_data into your model function.

## 0. Imports

In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from sklearn.preprocessing import StandardScaler

from smart_stethoscope.ml_logic.data_loading import load_data
from smart_stethoscope.ml_logic.preprocessing import (
    extract_mfcc_features,
    preprocess_tabular_data
)
from smart_stethoscope.ml_logic.model import run_logistic_baseline

In [2]:
os.chdir("/Users/keira/code/mi-mi-mia/smart-stethoscope")

## 1. Preprocess and split tabular data

In [3]:
# Load raw data
df = load_data()

df.shape


✅ Processed audio files already exist, skipping extraction

Load data from cached CSV...


(6898, 14)

In [4]:
X_train_tab, X_test_tab, y_train, y_test, train_pids, test_pids = preprocess_tabular_data(df)

print("X_train_tab shape:", X_train_tab.shape)
print("X_test_tab shape:", X_test_tab.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("Train patients:", len(train_pids))
print("Test patients:", len(test_pids))

X_train_tab shape: (5806, 15)
X_test_tab shape: (1054, 15)
y_train shape: (5806,)
y_test shape: (1054,)
Train patients: 98
Test patients: 25


/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [5]:
print(X_train_tab.columns.tolist())

['crackles', 'wheezes', 'filename', 'cycle_filename', 'age', 'sex', 'chest_location_Al', 'chest_location_Ar', 'chest_location_Ll', 'chest_location_Lr', 'chest_location_Pl', 'chest_location_Pr', 'chest_location_Tc', 'cycle_length', 'bmi']


## 2. Add MFCC Features 

In [6]:
# Extract MFCCs separately for train/test
cycle_audio_path = "preprocessed_data/audio_breathing_cycles"

train_mfcc = extract_mfcc_features(X_train_tab, cycle_audio_path)
test_mfcc = extract_mfcc_features(X_test_tab, cycle_audio_path)

print("train_mfcc shape:", train_mfcc.shape)
print("test_mfcc shape:", test_mfcc.shape)

train_mfcc shape: (5806, 53)
test_mfcc shape: (1054, 53)


In [7]:
# Merge MFCC features onto tabular train/test using cycle-level key
X_train_full = X_train_tab.merge(train_mfcc, on="cycle_filename", how="left")
X_test_full = X_test_tab.merge(test_mfcc, on="cycle_filename", how="left")

print("X_train_full shape:", X_train_full.shape)
print("X_test_full shape:", X_test_full.shape)

X_train_full shape: (5806, 67)
X_test_full shape: (1054, 67)


In [8]:
# Drop identifier columns before training
X_train_full = X_train_full.drop(columns=["filename", "cycle_filename"])
X_test_full = X_test_full.drop(columns=["filename", "cycle_filename"])

print("Final X_train_full shape:", X_train_full.shape)
print("Final X_test_full shape:", X_test_full.shape)
print("Missing values in X_train_full:", X_train_full.isna().sum().sum())
print("Missing values in X_test_full:", X_test_full.isna().sum().sum())

Final X_train_full shape: (5806, 65)
Final X_test_full shape: (1054, 65)
Missing values in X_train_full: 0
Missing values in X_test_full: 0


In [9]:
# Identify MFCC columns only
mfcc_cols = [col for col in X_train_full.columns if col.startswith("mfcc_")]

print("Number of MFCC columns:", len(mfcc_cols))
print(mfcc_cols[:5])

Number of MFCC columns: 52
['mfcc_1_mean', 'mfcc_2_mean', 'mfcc_3_mean', 'mfcc_4_mean', 'mfcc_5_mean']


In [10]:
# Scale MFCC features only, fit on train and apply to test
mfcc_scaler = StandardScaler()

X_train_full_scaled = X_train_full.copy()
X_test_full_scaled = X_test_full.copy()

X_train_full_scaled[mfcc_cols] = mfcc_scaler.fit_transform(X_train_full[mfcc_cols])
X_test_full_scaled[mfcc_cols] = mfcc_scaler.transform(X_test_full[mfcc_cols])

print("X_train_full_scaled shape:", X_train_full_scaled.shape)
print("X_test_full_scaled shape:", X_test_full_scaled.shape)

X_train_full_scaled shape: (5806, 65)
X_test_full_scaled shape: (1054, 65)


## 3. Train and evaluate model

In [11]:
model, y_pred = run_logistic_baseline(
    X_train_full_scaled,
    X_test_full_scaled,
    y_train,
    y_test
)

Accuracy: 0.7770398481973435
              precision    recall  f1-score   support

           0       0.43      0.37      0.40        57
           1       0.99      0.87      0.92       891
           2       0.10      0.09      0.10        43
           3       0.44      0.17      0.24        24
           4       0.11      0.65      0.19        26
           5       0.05      0.08      0.06        13

    accuracy                           0.78      1054
   macro avg       0.35      0.37      0.32      1054
weighted avg       0.88      0.78      0.82      1054



/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ 

👉 Logistic regression struggling numerically because of class imbalance + high dimensional correlated MFCC features.